# Features Transacciones VPC - SOLO CASH-IN (n3 general + n2 sub)
Tabla `HM_TRANSACCIONES_CASHIN_VPC` (panel RUC x periodo). **Todo el monto/trxs es cash-in.**

- **n3**: monto, trxs, monto_var1, monto_mm6, monto_tend6.
- **n2**: monto, trxs (34 sub-categorias).
- **Diversidad de cash-in**: `n_categorias_cashin` (n3 distintos), `n_subcategorias_cashin` (n2 distintos), `n_tipos_cashin` (desc distintos) + su var1 y mm6.
- Globales: tot_monto, tot_trxs, tot_monto_tend6, meses_activos_6m.

In [ ]:
# 1) Instalacion
%pip install -q awswrangler

In [ ]:
# 2) Imports y CONFIG
import time
import awswrangler as wr

DATABASE       = "disc_comercial"
WORKGROUP      = "ibk-discovery-comercial-work-group"
S3_OUTPUT      = "s3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/athena-results/"
PERIODO_INICIO = 202401
PERIODO_FIN    = 202605

In [ ]:
# 3) Utilidades
def run_ddl(sql: str, label: str = ""):
    qid = wr.athena.start_query_execution(sql=sql, database=DATABASE, workgroup=WORKGROUP, s3_output=S3_OUTPUT)
    res = wr.athena.wait_query(query_execution_id=qid)
    state = res["Status"]["State"]
    if state != "SUCCEEDED":
        raise RuntimeError("[" + label + "] " + state + ": " + str(res["Status"].get("StateChangeReason","")))
    print("  OK [" + label + "] qid=" + qid)
    return qid

def drop_table(name: str):
    run_ddl("DROP TABLE IF EXISTS " + DATABASE + "." + name, "drop " + name)

In [ ]:
# 4) Template SQL (solo CASH-IN, n3 + n2)
SQL_CASHIN = r"""
-- ============================================================================
-- FEATURES TRANSACCIONES VPC (SOLO CASH-IN) - TABLA ANCHA por N3 (general) y N2 (sub)
-- Fuente : e_perm_aws.t_agg_mes_vpc_transacc_sav_imp   (filtrado tipo_cash_desc='CASH-IN')
-- Cruce  : e_perm_aws.T_VPC_CLIENTE_BANCA_FINAL_HST (cuc -> ruc)
-- Grano  : num_ruc_val x periodo_val (panel balanceado). TODO el monto/trxs es CASH-IN.
-- Incluye cuantos tipos de cash-in DISTINTOS hace el cliente (n3, n2, desc).
-- ============================================================================
-- DROP TABLE IF EXISTS disc_comercial.HM_TRANSACCIONES_CASHIN_VPC

CREATE TABLE disc_comercial.HM_TRANSACCIONES_CASHIN_VPC
WITH (format='Parquet', parquet_compression='SNAPPY', partitioned_by=ARRAY['periodo']) AS (

WITH params AS (
    SELECT {periodo_inicio} AS periodo_inicio, {periodo_fin} AS periodo_fin
),
periodos AS (
    SELECT periodo_val FROM (
        SELECT (anio*100 + mes) AS periodo_val
        FROM (SELECT anio FROM UNNEST(SEQUENCE(2024, 2026)) AS t(anio)) a
        CROSS JOIN (SELECT mes FROM UNNEST(SEQUENCE(1, 12)) AS t(mes)) m
    ) g CROSS JOIN params p
    WHERE g.periodo_val BETWEEN p.periodo_inicio AND p.periodo_fin
),
cuc_ruc AS (
    SELECT periodo_val, num_ruc_val, cuc_num
    FROM e_perm_aws.T_VPC_CLIENTE_BANCA_FINAL_HST
    CROSS JOIN params
    WHERE fecha_dt IN (
        SELECT MAX(fecha_dt) FROM e_perm_aws.T_VPC_CLIENTE_BANCA_FINAL_HST
        CROSS JOIN params p2
        WHERE CAST(YEAR(fecha_dt)*100 + MONTH(fecha_dt) AS INTEGER) BETWEEN p2.periodo_inicio AND p2.periodo_fin
        GROUP BY CAST(YEAR(fecha_dt)*100 + MONTH(fecha_dt) AS INTEGER)
    )
    AND COALESCE(num_ruc_val,'.') NOT LIKE '.' AND COALESCE(num_ruc_val,'.') NOT LIKE ''
),
universo_ruc AS (
    SELECT DISTINCT r.num_ruc_val
    FROM e_perm_aws.t_agg_mes_vpc_transacc_sav_imp t
    INNER JOIN cuc_ruc r ON t.cuc_num_val = r.cuc_num AND t.periodo_val = r.periodo_val
    CROSS JOIN params p
    WHERE t.periodo_val BETWEEN p.periodo_inicio AND p.periodo_fin
      AND t.tipo_cash_desc = 'CASH-IN'
      AND r.num_ruc_val IS NOT NULL
),
malla AS ( SELECT u.num_ruc_val, pe.periodo_val FROM universo_ruc u CROSS JOIN periodos pe ),
base AS (
    SELECT t.periodo_val, r.num_ruc_val,
           t.grupo_final_vpc_n3   AS n3,
           t.grupo_final_vpc_n2   AS n2,
           t.grupo_final_vpc_desc AS desc4,
           t.cant_transacc,
           t.montotransaccion_solar_amt AS monto
    FROM e_perm_aws.t_agg_mes_vpc_transacc_sav_imp t
    INNER JOIN cuc_ruc r ON t.cuc_num_val = r.cuc_num AND t.periodo_val = r.periodo_val
    CROSS JOIN params p
    WHERE t.periodo_val BETWEEN p.periodo_inicio AND p.periodo_fin
      AND t.tipo_cash_desc = 'CASH-IN'
      AND r.num_ruc_val IS NOT NULL
),
agg AS (
    SELECT periodo_val, num_ruc_val,
        -- N3 (general) - solo CASH-IN,
        SUM(CASE WHEN n3='POS' THEN monto ELSE 0 END)              AS pos_monto,
        SUM(CASE WHEN n3='POS' THEN cant_transacc ELSE 0 END)      AS pos_trxs,
        SUM(CASE WHEN n3='RECAUDACION' THEN monto ELSE 0 END)              AS recaud_monto,
        SUM(CASE WHEN n3='RECAUDACION' THEN cant_transacc ELSE 0 END)      AS recaud_trxs,
        SUM(CASE WHEN n3='DEPOSITOS' THEN monto ELSE 0 END)              AS depos_monto,
        SUM(CASE WHEN n3='DEPOSITOS' THEN cant_transacc ELSE 0 END)      AS depos_trxs,
        SUM(CASE WHEN n3='PAGO DE SERVICIOS' THEN monto ELSE 0 END)              AS pag_serv_monto,
        SUM(CASE WHEN n3='PAGO DE SERVICIOS' THEN cant_transacc ELSE 0 END)      AS pag_serv_trxs,
        SUM(CASE WHEN n3='PAGOS MASIVOS' THEN monto ELSE 0 END)              AS pag_masiv_monto,
        SUM(CASE WHEN n3='PAGOS MASIVOS' THEN cant_transacc ELSE 0 END)      AS pag_masiv_trxs,
        SUM(CASE WHEN n3='PAGOS RECIBIDOS' THEN monto ELSE 0 END)              AS pag_recib_monto,
        SUM(CASE WHEN n3='PAGOS RECIBIDOS' THEN cant_transacc ELSE 0 END)      AS pag_recib_trxs,
        SUM(CASE WHEN n3='TRANSFERENCIAS' THEN monto ELSE 0 END)              AS transf_monto,
        SUM(CASE WHEN n3='TRANSFERENCIAS' THEN cant_transacc ELSE 0 END)      AS transf_trxs,
        SUM(CASE WHEN n3='COBRANZAS' THEN monto ELSE 0 END)              AS cobranz_monto,
        SUM(CASE WHEN n3='COBRANZAS' THEN cant_transacc ELSE 0 END)      AS cobranz_trxs,
        SUM(CASE WHEN n3='OPERACIONES MESA' THEN monto ELSE 0 END)              AS op_mesa_monto,
        SUM(CASE WHEN n3='OPERACIONES MESA' THEN cant_transacc ELSE 0 END)      AS op_mesa_trxs,
        SUM(CASE WHEN n3='REACTIVA' THEN monto ELSE 0 END)              AS reactiva_monto,
        SUM(CASE WHEN n3='REACTIVA' THEN cant_transacc ELSE 0 END)      AS reactiva_trxs,
        -- N2 (sub) - solo CASH-IN,
        SUM(CASE WHEN n3='TRANSFERENCIAS' AND n2='TRANSFERENCIA IBK' THEN monto ELSE 0 END)         AS transf_ibk_monto,
        SUM(CASE WHEN n3='TRANSFERENCIAS' AND n2='TRANSFERENCIA IBK' THEN cant_transacc ELSE 0 END) AS transf_ibk_trxs,
        SUM(CASE WHEN n3='TRANSFERENCIAS' AND n2='TRANFERENCIA EXTERIOR' THEN monto ELSE 0 END)         AS transf_exterior_monto,
        SUM(CASE WHEN n3='TRANSFERENCIAS' AND n2='TRANFERENCIA EXTERIOR' THEN cant_transacc ELSE 0 END) AS transf_exterior_trxs,
        SUM(CASE WHEN n3='TRANSFERENCIAS' AND n2='TRANSFERENCIA INTERBANCARIA' THEN monto ELSE 0 END)         AS transf_interbanc_monto,
        SUM(CASE WHEN n3='TRANSFERENCIAS' AND n2='TRANSFERENCIA INTERBANCARIA' THEN cant_transacc ELSE 0 END) AS transf_interbanc_trxs,
        SUM(CASE WHEN n3='RECAUDACION' AND n2='RECAUDACION PAGO AUTOMATICO' THEN monto ELSE 0 END)         AS recaud_pago_auto_monto,
        SUM(CASE WHEN n3='RECAUDACION' AND n2='RECAUDACION PAGO AUTOMATICO' THEN cant_transacc ELSE 0 END) AS recaud_pago_auto_trxs,
        SUM(CASE WHEN n3='RECAUDACION' AND n2='RECAUDACION MAE' THEN monto ELSE 0 END)         AS recaud_mae_monto,
        SUM(CASE WHEN n3='RECAUDACION' AND n2='RECAUDACION MAE' THEN cant_transacc ELSE 0 END) AS recaud_mae_trxs,
        SUM(CASE WHEN n3='RECAUDACION' AND n2='RECAUDACION' THEN monto ELSE 0 END)         AS recaud_gen_monto,
        SUM(CASE WHEN n3='RECAUDACION' AND n2='RECAUDACION' THEN cant_transacc ELSE 0 END) AS recaud_gen_trxs,
        SUM(CASE WHEN n3='REACTIVA' AND n2='REACTIVA' THEN monto ELSE 0 END)         AS reactiva_gen_monto,
        SUM(CASE WHEN n3='REACTIVA' AND n2='REACTIVA' THEN cant_transacc ELSE 0 END) AS reactiva_gen_trxs,
        SUM(CASE WHEN n3='POS' AND n2='VISA' THEN monto ELSE 0 END)         AS pos_visa_monto,
        SUM(CASE WHEN n3='POS' AND n2='VISA' THEN cant_transacc ELSE 0 END) AS pos_visa_trxs,
        SUM(CASE WHEN n3='POS' AND n2='MASTERCARD' THEN monto ELSE 0 END)         AS pos_master_monto,
        SUM(CASE WHEN n3='POS' AND n2='MASTERCARD' THEN cant_transacc ELSE 0 END) AS pos_master_trxs,
        SUM(CASE WHEN n3='POS' AND n2='AMEX' THEN monto ELSE 0 END)         AS pos_amex_monto,
        SUM(CASE WHEN n3='POS' AND n2='AMEX' THEN cant_transacc ELSE 0 END) AS pos_amex_trxs,
        SUM(CASE WHEN n3='POS' AND n2='DINERS' THEN monto ELSE 0 END)         AS pos_diners_monto,
        SUM(CASE WHEN n3='POS' AND n2='DINERS' THEN cant_transacc ELSE 0 END) AS pos_diners_trxs,
        SUM(CASE WHEN n3='PAGOS RECIBIDOS' AND n2='PROVEEDORES' THEN monto ELSE 0 END)         AS pag_recib_prov_monto,
        SUM(CASE WHEN n3='PAGOS RECIBIDOS' AND n2='PROVEEDORES' THEN cant_transacc ELSE 0 END) AS pag_recib_prov_trxs,
        SUM(CASE WHEN n3='PAGOS RECIBIDOS' AND n2='PLANILLAS' THEN monto ELSE 0 END)         AS pag_recib_plan_monto,
        SUM(CASE WHEN n3='PAGOS RECIBIDOS' AND n2='PLANILLAS' THEN cant_transacc ELSE 0 END) AS pag_recib_plan_trxs,
        SUM(CASE WHEN n3='PAGOS RECIBIDOS' AND n2='VARIOS' THEN monto ELSE 0 END)         AS pag_recib_varios_monto,
        SUM(CASE WHEN n3='PAGOS RECIBIDOS' AND n2='VARIOS' THEN cant_transacc ELSE 0 END) AS pag_recib_varios_trxs,
        SUM(CASE WHEN n3='PAGOS RECIBIDOS' AND n2='CTS' THEN monto ELSE 0 END)         AS pag_recib_cts_monto,
        SUM(CASE WHEN n3='PAGOS RECIBIDOS' AND n2='CTS' THEN cant_transacc ELSE 0 END) AS pag_recib_cts_trxs,
        SUM(CASE WHEN n3='PAGOS MASIVOS' AND n2='PROVEEDORES' THEN monto ELSE 0 END)         AS pag_masiv_prov_monto,
        SUM(CASE WHEN n3='PAGOS MASIVOS' AND n2='PROVEEDORES' THEN cant_transacc ELSE 0 END) AS pag_masiv_prov_trxs,
        SUM(CASE WHEN n3='PAGOS MASIVOS' AND n2='PLANILLAS' THEN monto ELSE 0 END)         AS pag_masiv_plan_monto,
        SUM(CASE WHEN n3='PAGOS MASIVOS' AND n2='PLANILLAS' THEN cant_transacc ELSE 0 END) AS pag_masiv_plan_trxs,
        SUM(CASE WHEN n3='PAGOS MASIVOS' AND n2='CTS' THEN monto ELSE 0 END)         AS pag_masiv_cts_monto,
        SUM(CASE WHEN n3='PAGOS MASIVOS' AND n2='CTS' THEN cant_transacc ELSE 0 END) AS pag_masiv_cts_trxs,
        SUM(CASE WHEN n3='PAGOS MASIVOS' AND n2='VARIOS' THEN monto ELSE 0 END)         AS pag_masiv_varios_monto,
        SUM(CASE WHEN n3='PAGOS MASIVOS' AND n2='VARIOS' THEN cant_transacc ELSE 0 END) AS pag_masiv_varios_trxs,
        SUM(CASE WHEN n3='PAGO DE SERVICIOS' AND n2='PAGO SUNAT' THEN monto ELSE 0 END)         AS pag_serv_sunat_monto,
        SUM(CASE WHEN n3='PAGO DE SERVICIOS' AND n2='PAGO SUNAT' THEN cant_transacc ELSE 0 END) AS pag_serv_sunat_trxs,
        SUM(CASE WHEN n3='PAGO DE SERVICIOS' AND n2='PAGO AFP' THEN monto ELSE 0 END)         AS pag_serv_afp_monto,
        SUM(CASE WHEN n3='PAGO DE SERVICIOS' AND n2='PAGO AFP' THEN cant_transacc ELSE 0 END) AS pag_serv_afp_trxs,
        SUM(CASE WHEN n3='PAGO DE SERVICIOS' AND n2='PAGO DE SERVICIOS' THEN monto ELSE 0 END)         AS pag_serv_gen_monto,
        SUM(CASE WHEN n3='PAGO DE SERVICIOS' AND n2='PAGO DE SERVICIOS' THEN cant_transacc ELSE 0 END) AS pag_serv_gen_trxs,
        SUM(CASE WHEN n3='OPERACIONES MESA' AND n2='OPERACIONES MESA' THEN monto ELSE 0 END)         AS op_mesa_gen_monto,
        SUM(CASE WHEN n3='OPERACIONES MESA' AND n2='OPERACIONES MESA' THEN cant_transacc ELSE 0 END) AS op_mesa_gen_trxs,
        SUM(CASE WHEN n3='OPERACIONES MESA' AND n2='DEPOSITOS A PLAZO' THEN monto ELSE 0 END)         AS op_mesa_dpf_monto,
        SUM(CASE WHEN n3='OPERACIONES MESA' AND n2='DEPOSITOS A PLAZO' THEN cant_transacc ELSE 0 END) AS op_mesa_dpf_trxs,
        SUM(CASE WHEN n3='OPERACIONES MESA' AND n2='CAMBIOS' THEN monto ELSE 0 END)         AS op_mesa_cambios_monto,
        SUM(CASE WHEN n3='OPERACIONES MESA' AND n2='CAMBIOS' THEN cant_transacc ELSE 0 END) AS op_mesa_cambios_trxs,
        SUM(CASE WHEN n3='OPERACIONES MESA' AND n2='TRANSFERENCIA INTERBANCARIA' THEN monto ELSE 0 END)         AS op_mesa_interbanc_monto,
        SUM(CASE WHEN n3='OPERACIONES MESA' AND n2='TRANSFERENCIA INTERBANCARIA' THEN cant_transacc ELSE 0 END) AS op_mesa_interbanc_trxs,
        SUM(CASE WHEN n3='DEPOSITOS' AND n2='ABONO POR CIERRE' THEN monto ELSE 0 END)         AS depos_abono_cierre_monto,
        SUM(CASE WHEN n3='DEPOSITOS' AND n2='ABONO POR CIERRE' THEN cant_transacc ELSE 0 END) AS depos_abono_cierre_trxs,
        SUM(CASE WHEN n3='DEPOSITOS' AND n2='REGULARIZACIONES ATM' THEN monto ELSE 0 END)         AS depos_regul_atm_monto,
        SUM(CASE WHEN n3='DEPOSITOS' AND n2='REGULARIZACIONES ATM' THEN cant_transacc ELSE 0 END) AS depos_regul_atm_trxs,
        SUM(CASE WHEN n3='DEPOSITOS' AND n2='CHEQUE' THEN monto ELSE 0 END)         AS depos_cheque_monto,
        SUM(CASE WHEN n3='DEPOSITOS' AND n2='CHEQUE' THEN cant_transacc ELSE 0 END) AS depos_cheque_trxs,
        SUM(CASE WHEN n3='DEPOSITOS' AND n2='EFECTIVO' THEN monto ELSE 0 END)         AS depos_efectivo_monto,
        SUM(CASE WHEN n3='DEPOSITOS' AND n2='EFECTIVO' THEN cant_transacc ELSE 0 END) AS depos_efectivo_trxs,
        SUM(CASE WHEN n3='DEPOSITOS' AND n2='REMESAS' THEN monto ELSE 0 END)         AS depos_remesas_monto,
        SUM(CASE WHEN n3='DEPOSITOS' AND n2='REMESAS' THEN cant_transacc ELSE 0 END) AS depos_remesas_trxs,
        SUM(CASE WHEN n3='COBRANZAS' AND n2='CC EXPORTACION' THEN monto ELSE 0 END)         AS cobranz_cc_export_monto,
        SUM(CASE WHEN n3='COBRANZAS' AND n2='CC EXPORTACION' THEN cant_transacc ELSE 0 END) AS cobranz_cc_export_trxs,
        SUM(CASE WHEN n3='COBRANZAS' AND n2='COBRANZAS EXPORTACION' THEN monto ELSE 0 END)         AS cobranz_export_monto,
        SUM(CASE WHEN n3='COBRANZAS' AND n2='COBRANZAS EXPORTACION' THEN cant_transacc ELSE 0 END) AS cobranz_export_trxs,
        SUM(CASE WHEN n3='COBRANZAS' AND n2='COBRANZAS DE DOCUMENTOS' THEN monto ELSE 0 END)         AS cobranz_docs_monto,
        SUM(CASE WHEN n3='COBRANZAS' AND n2='COBRANZAS DE DOCUMENTOS' THEN cant_transacc ELSE 0 END) AS cobranz_docs_trxs,
        SUM(monto)                       AS tot_monto,
        SUM(cant_transacc)               AS tot_trxs,
        -- cuantos tipos de CASH-IN DISTINTOS hace el cliente en el mes,
        COUNT(DISTINCT n3)               AS n_categorias_cashin,
        COUNT(DISTINCT n2)               AS n_subcategorias_cashin,
        COUNT(DISTINCT desc4)            AS n_tipos_cashin
    FROM base GROUP BY periodo_val, num_ruc_val
),
pivot AS (
    SELECT m.num_ruc_val, m.periodo_val,
        COALESCE(a.pos_monto,0) AS pos_monto,
        COALESCE(a.pos_trxs,0) AS pos_trxs,
        COALESCE(a.recaud_monto,0) AS recaud_monto,
        COALESCE(a.recaud_trxs,0) AS recaud_trxs,
        COALESCE(a.depos_monto,0) AS depos_monto,
        COALESCE(a.depos_trxs,0) AS depos_trxs,
        COALESCE(a.pag_serv_monto,0) AS pag_serv_monto,
        COALESCE(a.pag_serv_trxs,0) AS pag_serv_trxs,
        COALESCE(a.pag_masiv_monto,0) AS pag_masiv_monto,
        COALESCE(a.pag_masiv_trxs,0) AS pag_masiv_trxs,
        COALESCE(a.pag_recib_monto,0) AS pag_recib_monto,
        COALESCE(a.pag_recib_trxs,0) AS pag_recib_trxs,
        COALESCE(a.transf_monto,0) AS transf_monto,
        COALESCE(a.transf_trxs,0) AS transf_trxs,
        COALESCE(a.cobranz_monto,0) AS cobranz_monto,
        COALESCE(a.cobranz_trxs,0) AS cobranz_trxs,
        COALESCE(a.op_mesa_monto,0) AS op_mesa_monto,
        COALESCE(a.op_mesa_trxs,0) AS op_mesa_trxs,
        COALESCE(a.reactiva_monto,0) AS reactiva_monto,
        COALESCE(a.reactiva_trxs,0) AS reactiva_trxs,
        COALESCE(a.transf_ibk_monto,0) AS transf_ibk_monto,
        COALESCE(a.transf_ibk_trxs,0) AS transf_ibk_trxs,
        COALESCE(a.transf_exterior_monto,0) AS transf_exterior_monto,
        COALESCE(a.transf_exterior_trxs,0) AS transf_exterior_trxs,
        COALESCE(a.transf_interbanc_monto,0) AS transf_interbanc_monto,
        COALESCE(a.transf_interbanc_trxs,0) AS transf_interbanc_trxs,
        COALESCE(a.recaud_pago_auto_monto,0) AS recaud_pago_auto_monto,
        COALESCE(a.recaud_pago_auto_trxs,0) AS recaud_pago_auto_trxs,
        COALESCE(a.recaud_mae_monto,0) AS recaud_mae_monto,
        COALESCE(a.recaud_mae_trxs,0) AS recaud_mae_trxs,
        COALESCE(a.recaud_gen_monto,0) AS recaud_gen_monto,
        COALESCE(a.recaud_gen_trxs,0) AS recaud_gen_trxs,
        COALESCE(a.reactiva_gen_monto,0) AS reactiva_gen_monto,
        COALESCE(a.reactiva_gen_trxs,0) AS reactiva_gen_trxs,
        COALESCE(a.pos_visa_monto,0) AS pos_visa_monto,
        COALESCE(a.pos_visa_trxs,0) AS pos_visa_trxs,
        COALESCE(a.pos_master_monto,0) AS pos_master_monto,
        COALESCE(a.pos_master_trxs,0) AS pos_master_trxs,
        COALESCE(a.pos_amex_monto,0) AS pos_amex_monto,
        COALESCE(a.pos_amex_trxs,0) AS pos_amex_trxs,
        COALESCE(a.pos_diners_monto,0) AS pos_diners_monto,
        COALESCE(a.pos_diners_trxs,0) AS pos_diners_trxs,
        COALESCE(a.pag_recib_prov_monto,0) AS pag_recib_prov_monto,
        COALESCE(a.pag_recib_prov_trxs,0) AS pag_recib_prov_trxs,
        COALESCE(a.pag_recib_plan_monto,0) AS pag_recib_plan_monto,
        COALESCE(a.pag_recib_plan_trxs,0) AS pag_recib_plan_trxs,
        COALESCE(a.pag_recib_varios_monto,0) AS pag_recib_varios_monto,
        COALESCE(a.pag_recib_varios_trxs,0) AS pag_recib_varios_trxs,
        COALESCE(a.pag_recib_cts_monto,0) AS pag_recib_cts_monto,
        COALESCE(a.pag_recib_cts_trxs,0) AS pag_recib_cts_trxs,
        COALESCE(a.pag_masiv_prov_monto,0) AS pag_masiv_prov_monto,
        COALESCE(a.pag_masiv_prov_trxs,0) AS pag_masiv_prov_trxs,
        COALESCE(a.pag_masiv_plan_monto,0) AS pag_masiv_plan_monto,
        COALESCE(a.pag_masiv_plan_trxs,0) AS pag_masiv_plan_trxs,
        COALESCE(a.pag_masiv_cts_monto,0) AS pag_masiv_cts_monto,
        COALESCE(a.pag_masiv_cts_trxs,0) AS pag_masiv_cts_trxs,
        COALESCE(a.pag_masiv_varios_monto,0) AS pag_masiv_varios_monto,
        COALESCE(a.pag_masiv_varios_trxs,0) AS pag_masiv_varios_trxs,
        COALESCE(a.pag_serv_sunat_monto,0) AS pag_serv_sunat_monto,
        COALESCE(a.pag_serv_sunat_trxs,0) AS pag_serv_sunat_trxs,
        COALESCE(a.pag_serv_afp_monto,0) AS pag_serv_afp_monto,
        COALESCE(a.pag_serv_afp_trxs,0) AS pag_serv_afp_trxs,
        COALESCE(a.pag_serv_gen_monto,0) AS pag_serv_gen_monto,
        COALESCE(a.pag_serv_gen_trxs,0) AS pag_serv_gen_trxs,
        COALESCE(a.op_mesa_gen_monto,0) AS op_mesa_gen_monto,
        COALESCE(a.op_mesa_gen_trxs,0) AS op_mesa_gen_trxs,
        COALESCE(a.op_mesa_dpf_monto,0) AS op_mesa_dpf_monto,
        COALESCE(a.op_mesa_dpf_trxs,0) AS op_mesa_dpf_trxs,
        COALESCE(a.op_mesa_cambios_monto,0) AS op_mesa_cambios_monto,
        COALESCE(a.op_mesa_cambios_trxs,0) AS op_mesa_cambios_trxs,
        COALESCE(a.op_mesa_interbanc_monto,0) AS op_mesa_interbanc_monto,
        COALESCE(a.op_mesa_interbanc_trxs,0) AS op_mesa_interbanc_trxs,
        COALESCE(a.depos_abono_cierre_monto,0) AS depos_abono_cierre_monto,
        COALESCE(a.depos_abono_cierre_trxs,0) AS depos_abono_cierre_trxs,
        COALESCE(a.depos_regul_atm_monto,0) AS depos_regul_atm_monto,
        COALESCE(a.depos_regul_atm_trxs,0) AS depos_regul_atm_trxs,
        COALESCE(a.depos_cheque_monto,0) AS depos_cheque_monto,
        COALESCE(a.depos_cheque_trxs,0) AS depos_cheque_trxs,
        COALESCE(a.depos_efectivo_monto,0) AS depos_efectivo_monto,
        COALESCE(a.depos_efectivo_trxs,0) AS depos_efectivo_trxs,
        COALESCE(a.depos_remesas_monto,0) AS depos_remesas_monto,
        COALESCE(a.depos_remesas_trxs,0) AS depos_remesas_trxs,
        COALESCE(a.cobranz_cc_export_monto,0) AS cobranz_cc_export_monto,
        COALESCE(a.cobranz_cc_export_trxs,0) AS cobranz_cc_export_trxs,
        COALESCE(a.cobranz_export_monto,0) AS cobranz_export_monto,
        COALESCE(a.cobranz_export_trxs,0) AS cobranz_export_trxs,
        COALESCE(a.cobranz_docs_monto,0) AS cobranz_docs_monto,
        COALESCE(a.cobranz_docs_trxs,0) AS cobranz_docs_trxs,
        COALESCE(a.tot_monto,0) AS tot_monto,
        COALESCE(a.tot_trxs,0) AS tot_trxs,
        COALESCE(a.n_categorias_cashin,0) AS n_categorias_cashin,
        COALESCE(a.n_subcategorias_cashin,0) AS n_subcategorias_cashin,
        COALESCE(a.n_tipos_cashin,0) AS n_tipos_cashin,
        CASE WHEN a.num_ruc_val IS NOT NULL THEN 1 ELSE 0 END AS flag_activo_mes
    FROM malla m
    LEFT JOIN agg a ON m.num_ruc_val=a.num_ruc_val AND m.periodo_val=a.periodo_val
),
panel AS (
    SELECT
        pivot.*,
        pos_monto - LAG(pos_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                AS pos_monto_var1,
        AVG(CAST(pos_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)                AS pos_monto_mm6,
        pos_monto - AVG(CAST(pos_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)    AS pos_monto_tend6,
        recaud_monto - LAG(recaud_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                AS recaud_monto_var1,
        AVG(CAST(recaud_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)                AS recaud_monto_mm6,
        recaud_monto - AVG(CAST(recaud_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)    AS recaud_monto_tend6,
        depos_monto - LAG(depos_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                AS depos_monto_var1,
        AVG(CAST(depos_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)                AS depos_monto_mm6,
        depos_monto - AVG(CAST(depos_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)    AS depos_monto_tend6,
        pag_serv_monto - LAG(pag_serv_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                AS pag_serv_monto_var1,
        AVG(CAST(pag_serv_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)                AS pag_serv_monto_mm6,
        pag_serv_monto - AVG(CAST(pag_serv_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)    AS pag_serv_monto_tend6,
        pag_masiv_monto - LAG(pag_masiv_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                AS pag_masiv_monto_var1,
        AVG(CAST(pag_masiv_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)                AS pag_masiv_monto_mm6,
        pag_masiv_monto - AVG(CAST(pag_masiv_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)    AS pag_masiv_monto_tend6,
        pag_recib_monto - LAG(pag_recib_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                AS pag_recib_monto_var1,
        AVG(CAST(pag_recib_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)                AS pag_recib_monto_mm6,
        pag_recib_monto - AVG(CAST(pag_recib_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)    AS pag_recib_monto_tend6,
        transf_monto - LAG(transf_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                AS transf_monto_var1,
        AVG(CAST(transf_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)                AS transf_monto_mm6,
        transf_monto - AVG(CAST(transf_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)    AS transf_monto_tend6,
        cobranz_monto - LAG(cobranz_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                AS cobranz_monto_var1,
        AVG(CAST(cobranz_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)                AS cobranz_monto_mm6,
        cobranz_monto - AVG(CAST(cobranz_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)    AS cobranz_monto_tend6,
        op_mesa_monto - LAG(op_mesa_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                AS op_mesa_monto_var1,
        AVG(CAST(op_mesa_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)                AS op_mesa_monto_mm6,
        op_mesa_monto - AVG(CAST(op_mesa_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)    AS op_mesa_monto_tend6,
        reactiva_monto - LAG(reactiva_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                AS reactiva_monto_var1,
        AVG(CAST(reactiva_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)                AS reactiva_monto_mm6,
        reactiva_monto - AVG(CAST(reactiva_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)    AS reactiva_monto_tend6,
        tot_monto - LAG(tot_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                AS tot_monto_var1,
        tot_monto - AVG(CAST(tot_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)    AS tot_monto_tend6,
        n_tipos_cashin - LAG(n_tipos_cashin,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)      AS n_tipos_cashin_var1,
        AVG(CAST(n_tipos_cashin AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)           AS n_tipos_cashin_mm6,
        SUM(flag_activo_mes) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)                          AS meses_activos_6m
    FROM pivot
)
SELECT p.*, CAST(p.periodo_val AS VARCHAR) AS periodo
FROM panel p
)
"""

## Ejecutar

In [ ]:
# 5) Crear la tabla
t0 = time.time()
drop_table("HM_TRANSACCIONES_CASHIN_VPC")
run_ddl(SQL_CASHIN.format(periodo_inicio=PERIODO_INICIO, periodo_fin=PERIODO_FIN), "cashin")
print("Listo en " + str(round((time.time()-t0)/60,1)) + " min. Tabla: " + DATABASE + ".HM_TRANSACCIONES_CASHIN_VPC")